In [1]:
import itertools
from copy import deepcopy as copy
# from netgraph import InteactiveGraph # pip "install netgraph
from functools import reduce
from itertools import product
from operator import itemgetter

import cvxpy as cp
import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import scipy
from numpy.linalg import matrix_rank as rank
from scipy.linalg import block_diag, fractional_matrix_power
from utils.Adversary import Adversary
from utils.Conversions import hamming_dist as dist
from utils.Conversions import to_str, visualize
from utils.Problems import Problem, exact_k, threshold_k
import random
from utils.Solvers import (
    adv_solver,
    instance_mask,
    ket,
    partial,
    span_solver,
    type_mask,
)

import networkx as nx

In [89]:
def ham_weight(x):
    return np.sum([1 for a in x if a==1])
def poly_sdp(f, deg):
    eps = cp.Variable()
    n = np.int32(np.log2(f.shape[0]))
    a = cp.Variable(f.shape[0])
    F = scipy.linalg.hadamard(f.shape[0])/f.shape[0]
    constraints = [eps >= 0, f-eps <= F@a, F@a <= f+eps]
    strings = list(itertools.product([0,1], repeat=n))   
    counter = 0
    zero_vect = np.ones_like(f)
    for x in strings:
        if ham_weight(x) >= deg:
            zero_vect[counter] = 0
        counter += 1
    constraints += [cp.multiply(zero_vect, a)==a]
    opt_prob = cp.Problem(cp.Minimize(eps), constraints)
    opt_prob.solve(solver="SCS", verbose=True)
    return a.value
        

In [90]:
def total_or(n):
    strings = list(itertools.product([0,1], repeat=n))
    print(strings)
    f = np.zeros(2**n)
    for i, x in enumerate(strings):
        wt = ham_weight(x)
        b = int(wt > 0)
        f[i] = (-1)**b
    return f
f = total_or(5)


[(0, 0, 0, 0, 0), (0, 0, 0, 0, 1), (0, 0, 0, 1, 0), (0, 0, 0, 1, 1), (0, 0, 1, 0, 0), (0, 0, 1, 0, 1), (0, 0, 1, 1, 0), (0, 0, 1, 1, 1), (0, 1, 0, 0, 0), (0, 1, 0, 0, 1), (0, 1, 0, 1, 0), (0, 1, 0, 1, 1), (0, 1, 1, 0, 0), (0, 1, 1, 0, 1), (0, 1, 1, 1, 0), (0, 1, 1, 1, 1), (1, 0, 0, 0, 0), (1, 0, 0, 0, 1), (1, 0, 0, 1, 0), (1, 0, 0, 1, 1), (1, 0, 1, 0, 0), (1, 0, 1, 0, 1), (1, 0, 1, 1, 0), (1, 0, 1, 1, 1), (1, 1, 0, 0, 0), (1, 1, 0, 0, 1), (1, 1, 0, 1, 0), (1, 1, 0, 1, 1), (1, 1, 1, 0, 0), (1, 1, 1, 0, 1), (1, 1, 1, 1, 0), (1, 1, 1, 1, 1)]


In [91]:
a = poly_sdp(f, 10)
F = scipy.linalg.hadamard(f.shape[0])/f.shape[0]
print(np.round(a, 5))
print(np.round(F@a, 5))

(CVXPY) Jun 13 03:06:11 PM: Your problem has 33 variables, 97 constraints, and 0 parameters.
(CVXPY) Jun 13 03:06:11 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 13 03:06:11 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 13 03:06:11 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 13 03:06:11 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 13 03:06:11 PM: Compiling problem (target solver=SCS).
(CVXPY) Jun 13 03:06:11 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Jun 13 03:06:11 PM: Applying reduction Dcp2Cone
(CVXPY) Jun 13 03:06:11 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jun 13 03:06:11 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jun 13 03:06:11 PM: Applying reduction SCS
(CVXPY) Jun 13 03:06:11 PM: Finished problem compilation (took 3.886

                                     CVXPY                                     
                                     v1.6.6                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.7 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 33, constraints m: 97
cones: 	  z: primal zero / dual free vars: 32
	  l: linear vars: 65
settings: eps_abs:

In [158]:
def dual_poly(f, deg):
    eps = cp.Variable()
    n = np.int32(np.log2(f.shape[0]))
    bp = cp.Variable(f.shape[0])
    bm = cp.Variable(f.shape[0])
    b = bp - bm
    ba = bp + bm
    F = scipy.linalg.hadamard(f.shape[0])/f.shape[0]
    constraints = [b@f.T >= eps, cp.sum(ba)==1, eps>=0, bp>=0, bm>=0]
    strings = list(itertools.product([0,1], repeat=n)) 
    counter = 0
    zero_vect = np.ones_like(f)
    for x in strings:
        if ham_weight(x) < deg:
            zero_vect[counter] = 0
        counter += 1
    a = b.T@F
    constraints += [cp.multiply(zero_vect, a)==a]
    opt_prob = cp.Problem(cp.Minimize(eps), constraints)
    print(eps)
    opt_prob.solve(solver="SCS", verbose=True)
    print(b.value@f.T)
    print(ba.value, np.sum(ba.value))
    return b.value
    

In [159]:
b = dual_poly(f, 1)
b = np.round(b, 5)
print(b)
print(F@b)
print(np.linalg.norm(b,1))

(CVXPY) Jun 13 05:44:16 PM: Your problem has 65 variables, 99 constraints, and 0 parameters.
(CVXPY) Jun 13 05:44:16 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 13 05:44:16 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 13 05:44:16 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 13 05:44:16 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 13 05:44:16 PM: Compiling problem (target solver=SCS).
(CVXPY) Jun 13 05:44:16 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Jun 13 05:44:16 PM: Applying reduction Dcp2Cone
(CVXPY) Jun 13 05:44:16 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jun 13 05:44:16 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jun 13 05:44:16 PM: Applying reduction SCS
(CVXPY) Jun 13 05:44:16 PM: Finished problem compilation (took 7.952

var3577
                                     CVXPY                                     
                                     v1.6.6                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.7 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 97, constraints m: 163
cones: 	  z: primal zero / dual free vars: 32
	  l: linear vars: 131
settings